# Reason-Graph — FELM-Science reproduction with Llama-3-8B-Instruct

Runs our BTProp reproduction on Kaggle's free GPUs with the paper's own open-source model,
reading the model's real **True/False token probabilities** (the paper's confidence method).

**Before running — notebook settings (right-hand panel):**
1. *Accelerator*: **GPU T4 x2** (the model is 16 GB; it's split across both GPUs).
2. *Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add-ons → Secrets*: a secret named **`GITHUB_TOKEN`** (read-only token for the private repo),
   attached to this notebook.

Then **Save Version → Save & Run All**. It runs on Kaggle's servers (~4–6 h); you can close the
laptop. Results appear in the notebook's **Output** tab. Steps in `docs/project-log/07-running-on-kaggle.md`.

## 1. Settings

In [ ]:
BRANCH = "phase-2/tree-construction"  # branch to run -- must be pushed to GitHub
MODEL = "NousResearch/Meta-Llama-3-8B-Instruct"

## 2. Check the GPUs (should list two Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Download our code from GitHub
The token is read from Kaggle Secrets, used once, then removed from the saved git config so it never
ends up in the notebook's output files.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo = "github.com/Mahakisore7/Reason-Graph.git"
os.chdir("/kaggle/working")
subprocess.run(
    ["git", "clone", "--quiet", "--branch", BRANCH, f"https://{token}@{repo}"], check=True
)
subprocess.run(
    ["git", "-C", "Reason-Graph", "remote", "set-url", "origin", f"https://{repo}"], check=True
)
del token
os.chdir("/kaggle/working/Reason-Graph")
!git log --oneline -3

## 4. Install what Kaggle doesn't already have
Kaggle already ships PyTorch with GPU support, scikit-learn and pandas. We add the rest, and make sure
`transformers` is recent enough for our code.

In [ ]:
!pip install -q tenacity openai sentence-transformers accelerate "transformers>=4.56"
import os

os.environ["PYTHONPATH"] = "/kaggle/working/Reason-Graph"

## 5. Build the calibration table (fit half: 300 segments)
For each segment, one forward pass of Llama reads P(True) vs P(False). Then we count, per confidence
bucket, how many segments are actually true vs false. Takes a few minutes.

In [ ]:
!python -u scripts/fit_emission_model.py --backend hf --model {MODEL}

## 6. Test on the held-out half (310 segments)
For every test segment: build a belief tree with Llama (decomposition / premises / correction), label
links with the NLI model, run the tree math, and compare AUROC of *raw confidence* vs *after the tree*.
The long step (hours). Everything is cached in `llm_cache/`, so a re-run resumes where it stopped.

In [ ]:
!python -u scripts/evaluate_felm.py --backend hf --model {MODEL}

## 7. Summary

In [ ]:
import glob
import json

for path in sorted(glob.glob("results/*.json")):
    with open(path) as f:
        data = json.load(f)
    keys = [
        "llm",
        "confidence_method",
        "num_examples",
        "sample_size",
        "raw_confidence_auroc",
        "belief_tree_auroc",
    ]
    print(path)
    for k in keys:
        if k in data:
            print(f"   {k}: {data[k]}")

## 8. Package results for download
`reason_graph_results.zip` (results + the LLM answer cache) appears in the Output tab. Unzip it into the
repo on your laptop to analyse or re-plot without re-running anything.

In [ ]:
!cd /kaggle/working && zip -qr reason_graph_results.zip Reason-Graph/results Reason-Graph/llm_cache && ls -lh reason_graph_results.zip